# Lab 22: DPO Alignment for Vietnamese LLM (Fixed Version)
**Mục tiêu:** Chạy đầy đủ Pipeline SFT-mini -> DPO -> GGUF Export trên T4 GPU.

In [ ]:
!pip install unsloth vllm
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
import os
from pathlib import Path
import torch

# --- CONFIGURATION ---
BASE_MODEL = "unsloth/Qwen2.5-3B-bnb-4bit"
MAX_LEN = 1024
SFT_SLICE = 1000
PREF_SLICE = 1000
REPO_ROOT = Path("/content/lab22")
REPO_ROOT.mkdir(parents=True, exist_ok=True)

SFT_PATH = REPO_ROOT / "adapters" / "sft-mini"
DPO_PATH = REPO_ROOT / "adapters" / "dpo"
GGUF_PATH = REPO_ROOT / "gguf"
SCREENSHOT_PATH = REPO_ROOT / "submission" / "screenshots"

for d in [SFT_PATH, DPO_PATH, GGUF_PATH, SCREENSHOT_PATH]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✓ Workspace initialized at {REPO_ROOT}")

## Stage 1: SFT-mini Training

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_LEN,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model, r = 16, target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16, lora_dropout = 0, bias = "none", use_gradient_checkpointing = "unsloth"
)

dataset = load_dataset("vilm/vietnamese-alpaca-dataset-cleaned-52k", split=f"train[:{SFT_SLICE}]")

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = dataset,
    dataset_text_field = "instruction", max_seq_length = MAX_LEN,
    args = TrainingArguments(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
        max_steps = 60, learning_rate = 2e-4, fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(), logging_steps = 10, output_dir = "outputs",
    ),
)
trainer.train()
model.save_pretrained(str(SFT_PATH))
tokenizer.save_pretrained(str(SFT_PATH))
print("✓ SFT-mini completed.")

## Stage 2 & 3: DPO Alignment

In [ ]:
import gc
del model, trainer; gc.collect(); torch.cuda.empty_cache()

from trl import DPOTrainer, DPOConfig
from unsloth import FastLanguageModel
from peft import PeftModel

# 1. Load SFT model for DPO
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = str(SFT_PATH),
    max_seq_length = MAX_LEN,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(model, r = 16, target_modules = ["q_proj", "v_proj"], lora_alpha = 32)

# 2. Prepare Pref Data
from datasets import load_dataset
raw_pref = load_dataset("argilla/ultrafeedback-binarized-preferences-cleaned", split=f"train[:{PREF_SLICE}]")
def format_dpo(row):
    return {
        "prompt": row["prompt"],
        "chosen": row["chosen"][-1]["content"],
        "rejected": row["rejected"][-1]["content"]
    }
dataset_pref = raw_pref.map(format_dpo)

# 3. DPO Training
dpo_trainer = DPOTrainer(
    model = model, ref_model = None,
    args = DPOConfig(
        output_dir = "outputs-dpo", per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8, max_steps = 40, learning_rate = 5e-7,
        beta = 0.1, logging_steps = 5, fp16 = True,
    ),
    train_dataset = dataset_pref, tokenizer = tokenizer,
)
dpo_trainer.train()
model.save_pretrained(str(DPO_PATH))
print("✓ DPO completed.")

## Stage 4: Evaluation & Screenshots

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import json

# Save Metrics
metrics = {"final_train_loss": float(dpo_trainer.state.log_history[-1].get("loss", 0))}
with open(DPO_PATH / "dpo_metrics.json", "w") as f:
    json.dump(metrics, f)

# Plot Reward Curves
logs = pd.DataFrame(dpo_trainer.state.log_history)
if "rewards/chosen" in logs.columns:
    plt.figure(figsize=(10, 5))
    plt.plot(logs["step"], logs["rewards/chosen"], label="Chosen")
    plt.plot(logs["step"], logs["rewards/rejected"], label="Rejected")
    plt.title("DPO Reward Curves")
    plt.legend()
    plt.savefig(SCREENSHOT_PATH / "03-dpo-reward-curves.png")
    plt.show()
print("✓ Deliverables generated.")

## Stage 5: GGUF Export

In [ ]:
model.save_pretrained_gguf(
    str(GGUF_PATH),
    tokenizer,
    quantization_method = "q4_k_m",
)
print(f"✓ GGUF model saved to {GGUF_PATH}")